# Basin Mean Precipitation — Observed and Best-Approach Model
## Reference Period: 1995–2014

**Purpose:**  
Compute basin-level mean precipitation for the reference period (1995–2014)
for all four data sources:

- **Observed** — from station monthly data
- **Best Single Model** — from the best-ranked model's monthly data per basin
- **3-Model Ensemble** — from the basin-specific 3-model ensemble monthly data
- **6-Model Ensemble** — from the equal-weight 6-model ensemble monthly data

**Three outputs per source:**
- Annual mean (mm/yr) — sum of 12 monthly means averaged over 1995–2014
- Wet season mean (mm/season) — Oct–Mar, using hydrological year convention
- Dry season mean (mm/season) — Apr–Sep

**Wet season hydrological year:**  
Oct(Y-1)–Mar(Y) = wet season of year Y.  
So wet season 1995 = Oct 1994 + Nov 1994 + Dec 1994 + Jan–Mar 1995.  
The 20 wet seasons (1995–2014) therefore require data from Oct 1994 to Mar 2015.

**Method:**  
1. For each station: sum months within each year/season → annual/seasonal total  
2. Average totals across years (1995–2014) → station long-term mean  
3. Average station means across stations in each basin → basin mean

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\basin mean from station and best approach\`


## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
print(f"numpy  {np.__version__}  |  pandas  {pd.__version__}")


numpy  2.1.3  |  pandas  2.2.3


## 2. Configuration

In [2]:
# ── Monthly data directories ──────────────────────────────────────────────────
BASE = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments")

# Single model monthly CSVs — one sub-folder per model
SINGLE_MONTHLY_DIR = BASE / "monthly_data"

# 3-model ensemble monthly CSV
ENS3_MONTHLY_CSV = BASE / "ensemble data" / "monthly" / "monthly_ensemble_all_stations.csv"

# 6-model ensemble monthly CSV
ENS6_MONTHLY_CSV = BASE / "6ensemble data" / "monthly" / "monthly_6ens_all_stations.csv"

# Observed monthly CSV (from Notebook 02)
OBS_MONTHLY_CSV = BASE / "stations data" / "monthly" / "obs_monthly_all_stations.csv"

# Validation results — needed to identify best model per basin
RECOMMENDATION_CSV = BASE / "validation" / "comparison" / "basin_approach_recommendation.csv"
TABLE3_CSV         = BASE / "validation" / "single.model" / "table3_best_model_per_basin.csv"

# Station mapping — for basin order
STATION_MAPPING = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS\Station_Basin_Mapping.xlsx"
)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = BASE / "basin mean from station and best approach"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Period ────────────────────────────────────────────────────────────────────
PERIOD_START = 1995
PERIOD_END   = 2014

WET_MONTHS = [10, 11, 12, 1, 2, 3]   # Oct–Mar
DRY_MONTHS = [4,  5,  6,  7, 8, 9]   # Apr–Sep

# All 6 models
ALL_MODELS = [
    "CMCC-CM2-SR5", "CNRM-ESM2-1", "EC-Earth3-Veg",
    "IPSL-CM6A-LR", "MPI-ESM1-2-LR", "NorESM2-MM",
]

print("Configuration loaded.")
print(f"  Period       : {PERIOD_START}–{PERIOD_END}")
print(f"  Output dir   : {OUTPUT_DIR}")


Configuration loaded.
  Period       : 1995–2014
  Output dir   : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\basin mean from station and best approach


## 3. Load Basin Order and Best-Model Assignments

The recommended approach per basin (from Notebook 11) and the best single
model per basin (from Notebook 03 Table 3) are loaded to know which model's
monthly CSV to read for each basin.


In [3]:
# Basin order from station mapping
stations = pd.read_excel(STATION_MAPPING)
stations["Station_ID"] = stations["Station_ID"].astype(str).str.strip()
stations["Basin"]      = stations["Basin"].astype(str).str.strip()
basin_order = list(dict.fromkeys(stations["Basin"].tolist()))
print(f"Basins in study: {len(basin_order)}")
print(f"  {basin_order}")

# Best single model per basin (from Table 3)
table3 = pd.read_csv(TABLE3_CSV)
table3 = table3[~table3["Basin"].isin(["Mean","Std Dev","Median","IQR (Q25-Q75)",
                                         "IQR (Q1-Q3)"])].copy()
table3["Basin"] = table3["Basin"].astype(str).str.strip()
best_model_dict = dict(zip(table3["Basin"], table3["Best_Model"]))
print(f"Best single model per basin:")
for b, m in best_model_dict.items():
    print(f"  {b:<30} : {m}")

# Recommended approach per basin (from Notebook 11)
recommend = pd.read_csv(RECOMMENDATION_CSV)
recommend["Basin"] = recommend["Basin"].astype(str).str.strip()
recommend_dict = dict(zip(recommend["Basin"], recommend["Recommended_Approach"]))
print(f"Recommended approach per basin:")
for b, a in recommend_dict.items():
    print(f"  {b:<30} : {a}")


Basins in study: 12
  ['N.R.S.W', 'YARMOUK (JORDAN)', 'JORDAN VALLY (JORDAN)', 'AMMAN ZARQA (JORDAN)', 'S.R.S.W', 'D.S.R.S.W', 'MUJIB', 'W. ARABA NORTH', 'HASA', 'AZRAQ (JORDAN)', 'JAFER', 'HAMMAD']
Best single model per basin:
  N.R.S.W                        : NorESM2-MM
  YARMOUK (JORDAN)               : MPI-ESM1-2-LR
  JORDAN VALLY (JORDAN)          : MPI-ESM1-2-LR
  AMMAN ZARQA (JORDAN)           : MPI-ESM1-2-LR
  S.R.S.W                        : MPI-ESM1-2-LR
  D.S.R.S.W                      : IPSL-CM6A-LR
  MUJIB                          : MPI-ESM1-2-LR
  W. ARABA NORTH                 : IPSL-CM6A-LR
  HASA                           : MPI-ESM1-2-LR
  AZRAQ (JORDAN)                 : MPI-ESM1-2-LR
  JAFER                          : NorESM2-MM
  HAMMAD                         : MPI-ESM1-2-LR
Recommended approach per basin:
  N.R.S.W                        : 3-Model Ensemble
  YARMOUK (JORDAN)               : Best Single Model
  JORDAN VALLY (JORDAN)          : Best Single Model
  

## 4. Core Computation Functions

**`compute_basin_means(monthly_df)`**  
Given a monthly CSV DataFrame (columns: `Station_ID, Basin, Year, Month, pr_mm_month`),
computes for each basin:
- Annual mean: sum all 12 months per year → mean over 20 years → mean over stations
- Wet season: sum Oct–Mar per hydrological year → mean over 20 years → mean over stations
- Dry season: sum Apr–Sep per calendar year → mean over 20 years → mean over stations

**Hydrological year for wet season:**  
`hydro_year = year if month <= 9 else year + 1`  
Wet season year Y = Oct(Y-1) + Nov(Y-1) + Dec(Y-1) + Jan(Y) + Feb(Y) + Mar(Y)


In [4]:
def hydro_year(df):
    """Add HydroYear column: month<=9 -> same year; month>=10 -> year+1."""
    df = df.copy()
    df["HydroYear"] = df["Year"].where(df["Month"] <= 9, df["Year"] + 1)
    return df


def station_period_means(monthly_df, period_start=1995, period_end=2014):
    """
    Compute annual, wet-season, and dry-season period means per station.

    Parameters
    ----------
    monthly_df : DataFrame with columns Station_ID, Station_Name, Basin,
                 Year, Month, pr_mm_month
    period_start, period_end : int

    Returns
    -------
    DataFrame with one row per station:
        Station_ID, Station_Name, Basin,
        Annual_Mean, Wet_Mean, Dry_Mean,
        N_Yrs_Annual, N_Yrs_Wet, N_Yrs_Dry
    """
    df = monthly_df.copy()
    df["pr_mm_month"] = pd.to_numeric(df["pr_mm_month"], errors="coerce")
    df["Year"]  = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    df = hydro_year(df)

    results = []
    for (sid, sname, basin), grp in df.groupby(
            ["Station_ID", "Station_Name", "Basin"], sort=False):

        # ── Annual: sum 12 months per calendar year, mean over period ─────────
        ann = grp[grp["Year"].between(period_start, period_end)]
        ann_yr = ann.groupby("Year")["pr_mm_month"].sum()
        # Only years with all 12 months (allow partial: sum all available)
        ann_yr = ann_yr[ann_yr.index.isin(range(period_start, period_end + 1))]
        annual_mean = ann_yr.mean() if len(ann_yr) > 0 else np.nan
        n_annual    = len(ann_yr)

        # ── Wet season: sum Oct–Mar per hydro-year, mean over period ──────────
        wet = grp[grp["Month"].isin(WET_MONTHS) &
                  grp["HydroYear"].between(period_start, period_end)]
        wet_yr = wet.groupby("HydroYear")["pr_mm_month"].sum()
        wet_yr = wet_yr[wet_yr.index.isin(range(period_start, period_end + 1))]
        wet_mean = wet_yr.mean() if len(wet_yr) > 0 else np.nan
        n_wet    = len(wet_yr)

        # ── Dry season: sum Apr–Sep per calendar year, mean over period ───────
        dry = grp[grp["Month"].isin(DRY_MONTHS) &
                  grp["Year"].between(period_start, period_end)]
        dry_yr = dry.groupby("Year")["pr_mm_month"].sum()
        dry_yr = dry_yr[dry_yr.index.isin(range(period_start, period_end + 1))]
        dry_mean = dry_yr.mean() if len(dry_yr) > 0 else np.nan
        n_dry    = len(dry_yr)

        results.append({
            "Station_ID"  : sid,
            "Station_Name": sname,
            "Basin"       : basin,
            "Annual_Mean" : round(annual_mean, 2) if not np.isnan(annual_mean) else np.nan,
            "Wet_Mean"    : round(wet_mean,    2) if not np.isnan(wet_mean)    else np.nan,
            "Dry_Mean"    : round(dry_mean,    2) if not np.isnan(dry_mean)    else np.nan,
            "N_Yrs_Annual": n_annual,
            "N_Yrs_Wet"   : n_wet,
            "N_Yrs_Dry"   : n_dry,
        })

    return pd.DataFrame(results)


def basin_means_from_stations(station_means_df, basin_order):
    """
    Average station-level means across stations within each basin.

    Returns DataFrame: Basin, Annual_Mean, Wet_Mean, Dry_Mean, N_Stations
    """
    rows = []
    all_basins = station_means_df["Basin"].unique()

    for basin in basin_order:
        if basin not in all_basins:
            rows.append({"Basin": basin,
                         "Annual_Mean": np.nan, "Wet_Mean": np.nan,
                         "Dry_Mean": np.nan, "N_Stations": 0})
            continue

        grp = station_means_df[station_means_df["Basin"] == basin]
        ann = grp["Annual_Mean"].dropna()
        wet = grp["Wet_Mean"].dropna()
        dry = grp["Dry_Mean"].dropna()

        rows.append({
            "Basin"       : basin,
            "Annual_Mean" : round(ann.mean(), 1) if len(ann) > 0 else np.nan,
            "Wet_Mean"    : round(wet.mean(), 1) if len(wet) > 0 else np.nan,
            "Dry_Mean"    : round(dry.mean(), 1) if len(dry) > 0 else np.nan,
            "N_Stations"  : len(grp),
        })

    return pd.DataFrame(rows)


print("Functions defined: station_period_means(), basin_means_from_stations()")


Functions defined: station_period_means(), basin_means_from_stations()


## 5. Observed Basin Means

Read from `stations data/monthly/obs_monthly_all_stations.csv` —
the monthly totals (mm/month) aggregated from daily station records in Notebook 02.


In [5]:
print("Loading observed monthly data ...")
obs_monthly = pd.read_csv(OBS_MONTHLY_CSV)
obs_monthly["Station_ID"] = obs_monthly["Station_ID"].astype(str).str.strip()
obs_monthly["Basin"]      = obs_monthly["Basin"].astype(str).str.strip()
print(f"  Rows: {len(obs_monthly):,}  |  Stations: {obs_monthly['Station_ID'].nunique()}")

obs_stn_means  = station_period_means(obs_monthly)
obs_basin_means = basin_means_from_stations(obs_stn_means, basin_order)

# Save station-level
obs_stn_means.to_csv(OUTPUT_DIR / "obs_station_period_means.csv", index=False)
obs_basin_means.to_csv(OUTPUT_DIR / "obs_basin_period_means.csv", index=False)

print("Observed basin means (Annual | Wet | Dry  mm):")
print(f"{'Basin':<28} {'Annual':>8} {'Wet':>8} {'Dry':>6} {'N_Stn':>6}")
print("-" * 60)
for _, r in obs_basin_means.iterrows():
    ann = f"{r['Annual_Mean']:.1f}" if not np.isnan(r['Annual_Mean']) else "—"
    wet = f"{r['Wet_Mean']:.1f}"    if not np.isnan(r['Wet_Mean'])    else "—"
    dry = f"{r['Dry_Mean']:.1f}"    if not np.isnan(r['Dry_Mean'])    else "—"
    print(f"  {r['Basin']:<26} {ann:>8} {wet:>8} {dry:>6} {r['N_Stations']:>6}")


Loading observed monthly data ...
  Rows: 11,760  |  Stations: 49
Observed basin means (Annual | Wet | Dry  mm):
Basin                          Annual      Wet    Dry  N_Stn
------------------------------------------------------------
  N.R.S.W                       445.0    419.6   25.4      4
  YARMOUK (JORDAN)              303.4    285.2   18.2      5
  JORDAN VALLY (JORDAN)         276.6    259.6   17.0      3
  AMMAN ZARQA (JORDAN)          222.1    211.8   10.3      6
  S.R.S.W                       349.2    331.5   17.7      4
  D.S.R.S.W                     254.8    239.6   15.2      7
  MUJIB                         172.3    161.4   10.9      6
  W. ARABA NORTH                185.4    173.5   12.0      3
  HASA                          162.7    150.0   12.7      1
  AZRAQ (JORDAN)                 52.2     48.2    4.0      4
  JAFER                          77.0     71.5    5.5      5
  HAMMAD                         67.5     55.6   11.9      1


## 6. Best Single Model Basin Means

For each basin, reads the monthly CSV of the best-ranked model (from Table 3)
from `monthly_data/<MODEL_NAME>/monthly_pr_all_stations.csv`.


In [6]:
print("Loading single model monthly data ...")

# Load all models into cache
model_monthly_cache = {}
for model in ALL_MODELS:
    csv_path = SINGLE_MONTHLY_DIR / model / "monthly_pr_all_stations.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df["Station_ID"] = df["Station_ID"].astype(str).str.strip()
        df["Basin"]      = df["Basin"].astype(str).str.strip()
        model_monthly_cache[model] = df
        print(f"  Loaded {model}: {len(df):,} rows")
    else:
        print(f"  [MISSING] {model}: {csv_path}")

print(f"Models loaded: {list(model_monthly_cache.keys())}")

# For each basin, use its best model
best_stn_rows = []
for basin in basin_order:
    best_model = best_model_dict.get(basin)
    if best_model is None or best_model not in model_monthly_cache:
        print(f"  [SKIP] {basin}: no best model or model not loaded")
        continue

    df_model = model_monthly_cache[best_model]
    basin_df = df_model[df_model["Basin"] == basin]
    if basin_df.empty:
        print(f"  [EMPTY] {basin} in {best_model}")
        continue

    stn_means = station_period_means(basin_df)
    stn_means["Best_Model"] = best_model
    best_stn_rows.append(stn_means)

best_stn_df = pd.concat(best_stn_rows, ignore_index=True) if best_stn_rows else pd.DataFrame()
best_basin_means = basin_means_from_stations(best_stn_df, basin_order)
# Add best model name
best_basin_means["Best_Model"] = best_basin_means["Basin"].map(best_model_dict)

best_stn_df.to_csv(OUTPUT_DIR / "single_best_station_period_means.csv", index=False)
best_basin_means.to_csv(OUTPUT_DIR / "single_best_basin_period_means.csv", index=False)

print("Best single model basin means:")
print(f"{'Basin':<28} {'Model':<22} {'Annual':>8} {'Wet':>8} {'Dry':>6}")
print("-" * 75)
for _, r in best_basin_means.iterrows():
    ann = f"{r['Annual_Mean']:.1f}" if not np.isnan(r['Annual_Mean']) else "—"
    wet = f"{r['Wet_Mean']:.1f}"    if not np.isnan(r['Wet_Mean'])    else "—"
    dry = f"{r['Dry_Mean']:.1f}"    if not np.isnan(r['Dry_Mean'])    else "—"
    mdl = str(r.get('Best_Model','?'))
    print(f"  {r['Basin']:<26} {mdl:<22} {ann:>8} {wet:>8} {dry:>6}")


Loading single model monthly data ...
  Loaded CMCC-CM2-SR5: 11,760 rows
  Loaded CNRM-ESM2-1: 11,760 rows
  Loaded EC-Earth3-Veg: 11,760 rows
  Loaded IPSL-CM6A-LR: 11,760 rows
  Loaded MPI-ESM1-2-LR: 11,760 rows
  Loaded NorESM2-MM: 11,760 rows
Models loaded: ['CMCC-CM2-SR5', 'CNRM-ESM2-1', 'EC-Earth3-Veg', 'IPSL-CM6A-LR', 'MPI-ESM1-2-LR', 'NorESM2-MM']
Best single model basin means:
Basin                        Model                    Annual      Wet    Dry
---------------------------------------------------------------------------
  N.R.S.W                    NorESM2-MM                399.8    367.0   27.8
  YARMOUK (JORDAN)           MPI-ESM1-2-LR             346.4    318.1   24.5
  JORDAN VALLY (JORDAN)      MPI-ESM1-2-LR             387.9    360.8   23.0
  AMMAN ZARQA (JORDAN)       MPI-ESM1-2-LR             230.8    215.4   13.5
  S.R.S.W                    MPI-ESM1-2-LR             294.5    277.2   15.0
  D.S.R.S.W                  IPSL-CM6A-LR              242.5    219.5   1

## 7. 3-Model Ensemble Basin Means

Reads `ensemble data/monthly/monthly_ensemble_all_stations.csv` — the
basin-specific 3-model ensemble monthly totals from Notebook 06.


In [7]:
print("Loading 3-model ensemble monthly data ...")
ens3_monthly = pd.read_csv(ENS3_MONTHLY_CSV)
ens3_monthly["Station_ID"] = ens3_monthly["Station_ID"].astype(str).str.strip()
ens3_monthly["Basin"]      = ens3_monthly["Basin"].astype(str).str.strip()
print(f"  Rows: {len(ens3_monthly):,}  |  Stations: {ens3_monthly['Station_ID'].nunique()}")

ens3_stn_means   = station_period_means(ens3_monthly)
ens3_basin_means = basin_means_from_stations(ens3_stn_means, basin_order)

ens3_stn_means.to_csv(OUTPUT_DIR / "ens3_station_period_means.csv", index=False)
ens3_basin_means.to_csv(OUTPUT_DIR / "ens3_basin_period_means.csv", index=False)

print("3-model ensemble basin means:")
print(f"{'Basin':<28} {'Annual':>8} {'Wet':>8} {'Dry':>6}")
print("-" * 55)
for _, r in ens3_basin_means.iterrows():
    ann = f"{r['Annual_Mean']:.1f}" if not np.isnan(r['Annual_Mean']) else "—"
    wet = f"{r['Wet_Mean']:.1f}"    if not np.isnan(r['Wet_Mean'])    else "—"
    dry = f"{r['Dry_Mean']:.1f}"    if not np.isnan(r['Dry_Mean'])    else "—"
    print(f"  {r['Basin']:<26} {ann:>8} {wet:>8} {dry:>6}")


Loading 3-model ensemble monthly data ...
  Rows: 11,760  |  Stations: 49
3-model ensemble basin means:
Basin                          Annual      Wet    Dry
-------------------------------------------------------
  N.R.S.W                       398.6    366.8   26.9
  YARMOUK (JORDAN)              307.8    281.5   22.2
  JORDAN VALLY (JORDAN)         247.4    228.6   15.8
  AMMAN ZARQA (JORDAN)          237.9    220.8   14.5
  S.R.S.W                       304.6    284.8   16.2
  D.S.R.S.W                     229.0    211.5   14.7
  MUJIB                         192.4    177.9   12.8
  W. ARABA NORTH                149.2    137.2   10.6
  HASA                          152.5    138.7   12.4
  AZRAQ (JORDAN)                104.1     90.8   12.1
  JAFER                          95.1     85.6    8.7
  HAMMAD                        101.2     80.4   19.0


## 8. 6-Model Ensemble Basin Means

Reads `6ensemble data/monthly/monthly_6ens_all_stations.csv` — the
equal-weight 6-model ensemble monthly totals from Notebook 09.


In [8]:
print("Loading 6-model ensemble monthly data ...")
ens6_monthly = pd.read_csv(ENS6_MONTHLY_CSV)
ens6_monthly["Station_ID"] = ens6_monthly["Station_ID"].astype(str).str.strip()
ens6_monthly["Basin"]      = ens6_monthly["Basin"].astype(str).str.strip()
print(f"  Rows: {len(ens6_monthly):,}  |  Stations: {ens6_monthly['Station_ID'].nunique()}")

ens6_stn_means   = station_period_means(ens6_monthly)
ens6_basin_means = basin_means_from_stations(ens6_stn_means, basin_order)

ens6_stn_means.to_csv(OUTPUT_DIR / "ens6_station_period_means.csv", index=False)
ens6_basin_means.to_csv(OUTPUT_DIR / "ens6_basin_period_means.csv", index=False)

print("6-model ensemble basin means:")
print(f"{'Basin':<28} {'Annual':>8} {'Wet':>8} {'Dry':>6}")
print("-" * 55)
for _, r in ens6_basin_means.iterrows():
    ann = f"{r['Annual_Mean']:.1f}" if not np.isnan(r['Annual_Mean']) else "—"
    wet = f"{r['Wet_Mean']:.1f}"    if not np.isnan(r['Wet_Mean'])    else "—"
    dry = f"{r['Dry_Mean']:.1f}"    if not np.isnan(r['Dry_Mean'])    else "—"
    print(f"  {r['Basin']:<26} {ann:>8} {wet:>8} {dry:>6}")


Loading 6-model ensemble monthly data ...
  Rows: 11,760  |  Stations: 49
6-model ensemble basin means:
Basin                          Annual      Wet    Dry
-------------------------------------------------------
  N.R.S.W                       401.3    366.6   27.1
  YARMOUK (JORDAN)              315.4    286.2   22.9
  JORDAN VALLY (JORDAN)         251.4    230.5   16.2
  AMMAN ZARQA (JORDAN)          243.2    222.6   16.8
  S.R.S.W                       310.0    287.5   17.5
  D.S.R.S.W                     230.8    214.4   13.2
  MUJIB                         197.2    181.6   13.0
  W. ARABA NORTH                150.4    137.2   10.8
  HASA                          154.2    137.7   13.7
  AZRAQ (JORDAN)                112.1     97.3   12.5
  JAFER                         101.8     91.7    8.5
  HAMMAD                        110.8     89.2   19.5


## 9. Identify Best-Approach Values per Basin

For the final comparison table, each basin's "Best Approach" values come
from whichever approach was recommended in Notebook 11 composite ranking.


In [9]:
def get_basin_val(df, basin, col):
    """Safely extract one scalar value from a basin means DataFrame."""
    mask = df["Basin"] == basin
    if not mask.any():
        return np.nan
    v = df.loc[mask, col].iloc[0]
    return v if not (isinstance(v, float) and np.isnan(v)) else np.nan

best_approach_rows = []
for basin in basin_order:
    appr = recommend_dict.get(basin, "Best Single Model")

    if appr == "Best Single Model":
        ann = get_basin_val(best_basin_means, basin, "Annual_Mean")
        wet = get_basin_val(best_basin_means, basin, "Wet_Mean")
        dry = get_basin_val(best_basin_means, basin, "Dry_Mean")
        model_label = best_model_dict.get(basin, "?")
    elif appr == "3-Model Ensemble":
        ann = get_basin_val(ens3_basin_means, basin, "Annual_Mean")
        wet = get_basin_val(ens3_basin_means, basin, "Wet_Mean")
        dry = get_basin_val(ens3_basin_means, basin, "Dry_Mean")
        model_label = "Top-3 Basin-Specific Ensemble"
    else:  # 6-Model Ensemble
        ann = get_basin_val(ens6_basin_means, basin, "Annual_Mean")
        wet = get_basin_val(ens6_basin_means, basin, "Wet_Mean")
        dry = get_basin_val(ens6_basin_means, basin, "Dry_Mean")
        model_label = "6-Model Equal-Weight Ensemble"

    best_approach_rows.append({
        "Basin"               : basin,
        "Recommended_Approach": appr,
        "Model_Label"         : model_label,
        "BestAppr_Annual"     : round(ann, 1) if not np.isnan(ann) else np.nan,
        "BestAppr_Wet"        : round(wet, 1) if not np.isnan(wet) else np.nan,
        "BestAppr_Dry"        : round(dry, 1) if not np.isnan(dry) else np.nan,
    })

best_appr_df = pd.DataFrame(best_approach_rows)
print("Best-approach means identified per basin.")
print(f"{'Basin':<28} {'Approach':<22} {'Annual':>8} {'Wet':>8} {'Dry':>6}")
print("-" * 78)
for _, r in best_appr_df.iterrows():
    ann = f"{r['BestAppr_Annual']:.1f}" if not (isinstance(r['BestAppr_Annual'],float) and np.isnan(r['BestAppr_Annual'])) else "—"
    wet = f"{r['BestAppr_Wet']:.1f}"    if not (isinstance(r['BestAppr_Wet'],float) and np.isnan(r['BestAppr_Wet'])) else "—"
    dry = f"{r['BestAppr_Dry']:.1f}"    if not (isinstance(r['BestAppr_Dry'],float) and np.isnan(r['BestAppr_Dry'])) else "—"
    print(f"  {r['Basin']:<26} {r['Recommended_Approach']:<22} {ann:>8} {wet:>8} {dry:>6}")


Best-approach means identified per basin.
Basin                        Approach                 Annual      Wet    Dry
------------------------------------------------------------------------------
  N.R.S.W                    3-Model Ensemble          398.6    366.8   26.9
  YARMOUK (JORDAN)           Best Single Model         346.4    318.1   24.5
  JORDAN VALLY (JORDAN)      Best Single Model         387.9    360.8   23.0
  AMMAN ZARQA (JORDAN)       Best Single Model         230.8    215.4   13.5
  S.R.S.W                    Best Single Model         294.5    277.2   15.0
  D.S.R.S.W                  3-Model Ensemble          229.0    211.5   14.7
  MUJIB                      Best Single Model         184.1    173.6    9.4
  W. ARABA NORTH             3-Model Ensemble          149.2    137.2   10.6
  HASA                       Best Single Model         143.8    131.7   11.0
  AZRAQ (JORDAN)             3-Model Ensemble          104.1     90.8   12.1
  JAFER                      Bes

## 10. Build Excel Workbook

Four sheets:
1. **Main Comparison** — Observed vs Best Approach (Annual + Wet + Dry)
2. **All Four Sources** — All sources side by side for each basin
3. **Station Detail** — Station-level annual/wet/dry means per source
4. **Aridity Context** — Basin means with aridity classification


In [10]:
# ── Style helpers ─────────────────────────────────────────────────────────────
def fill(hex_color):
    return PatternFill("solid", start_color=hex_color)

thin  = Side(style="thin",   color="AAAAAA")
med   = Side(style="medium", color="777777")
BORDER  = Border(left=thin, right=thin, top=thin, bottom=thin)
BORDER_M= Border(left=med,  right=med,  top=med,  bottom=med)
CENTER  = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT    = Alignment(horizontal="left",   vertical="center")

# Colour palette
CB  = "1F4E79"   # header blue
CG  = "145A32"   # header green (observed)
CO  = "784212"   # header orange-brown (model)
CP  = "4A235B"   # header purple (best approach)
CT  = "D9E1F2"   # title background
CLB = "D6EAF8"   # light blue (observed data cells)
CLG = "D5F5E3"   # light green (single model cells)
CLO = "FAE5D3"   # light orange (3-ens cells)
CLY = "FFF9C4"   # light yellow (6-ens / best approach cells)
CLR = "FDEDEC"   # light red (no data)
CLW = "FDFEFE"   # near white

def hdr(cell, text, bg=CB, fg="FFFFFF", sz=9, wrap=True):
    cell.value     = text
    cell.fill      = fill(bg)
    cell.font      = Font(name="Arial", bold=True, color=fg, size=sz)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=wrap)
    cell.border    = BORDER

def dat(cell, value="", bold=False, bg=None, align=CENTER, fmt=None):
    cell.value     = value
    cell.font      = Font(name="Arial", bold=bold, size=9)
    cell.alignment = align
    cell.border    = BORDER
    if bg: cell.fill = fill(bg)
    if fmt and isinstance(value, (int,float)) and        not (isinstance(value, float) and np.isnan(value)):
        cell.number_format = fmt

def title_row(ws, text, n_cols, row=1):
    ws.merge_cells(f"A{row}:{get_column_letter(n_cols)}{row}")
    c = ws[f"A{row}"]
    c.value     = text
    c.font      = Font(name="Arial", bold=True, size=12, color=CB)
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.fill      = fill(CT)
    ws.row_dimensions[row].height = 22

def grp_header(ws, row, cols_start, cols_end, label, bg):
    ws.merge_cells(start_row=row, start_column=cols_start,
                   end_row=row,   end_column=cols_end)
    c = ws.cell(row=row, column=cols_start, value=label)
    c.fill = fill(bg); c.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    c.alignment = CENTER; c.border = BORDER

def nv(v, default=""):
    if v is None: return default
    try:
        if np.isnan(float(v)): return default
    except (TypeError, ValueError): pass
    return v

def fmt_num(v, decimals=1):
    """Format number or return em dash."""
    if v is None or v == "": return "—"
    try:
        if np.isnan(float(v)): return "—"
        return round(float(v), decimals)
    except (TypeError, ValueError):
        return "—"

NUM = "0.0"

APPROACH_BG = {
    "Best Single Model": CLG,
    "3-Model Ensemble" : CLO,
    "6-Model Ensemble" : CLY,
}

wb  = Workbook()

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 1 — Main Comparison: Observed vs Best Approach
# ══════════════════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = "Observed vs Best Approach"
N1 = 9

title_row(ws1,
    f"Basin Mean Precipitation — Observed Stations vs Best-Approach Model | {PERIOD_START}–{PERIOD_END}",
    N1)

grp_header(ws1, 2, 1, 3, "BASIN INFO",                     CB)
grp_header(ws1, 2, 4, 6, "OBSERVED — Station Mean (mm)",   CG)
grp_header(ws1, 2, 7, 9, "BEST-APPROACH MODEL (mm)",       CO)
ws1.row_dimensions[2].height = 16

h1 = [("Basin",CB),("NStations",CB),("RecommendedApproach",CB),
      ("Annual(mm/yr)",CG),("Wet Season(mm)",CG),("Dry Season(mm)",CG),
      ("Model / Ensemble",CO),("Annual(mm/yr)",CO),("Wet Season(mm)",CO)]
# Only 9 cols
for ci,(h,bg) in enumerate(h1[:N1],1): hdr(ws1.cell(row=3,column=ci),h,bg=bg)
ws1.row_dimensions[3].height = 36

for i,w in enumerate([26,9,22,13,13,11,26,13,13],1):
    ws1.column_dimensions[get_column_letter(i)].width = w
ws1.freeze_panes = "A4"

for ri,basin in enumerate(basin_order):
    r = ri + 4
    appr   = recommend_dict.get(basin,"Best Single Model")
    abg    = APPROACH_BG.get(appr,CLY)
    n_stn  = nv(get_basin_val(obs_basin_means, basin, "N_Stations"), 0)
    obs_ann= fmt_num(get_basin_val(obs_basin_means, basin, "Annual_Mean"))
    obs_wet= fmt_num(get_basin_val(obs_basin_means, basin, "Wet_Mean"))
    obs_dry= fmt_num(get_basin_val(obs_basin_means, basin, "Dry_Mean"))
    ba_row = best_appr_df[best_appr_df["Basin"]==basin]
    if not ba_row.empty:
        mlabel  = ba_row["Model_Label"].iloc[0]
        ba_ann  = fmt_num(ba_row["BestAppr_Annual"].iloc[0])
        ba_wet  = fmt_num(ba_row["BestAppr_Wet"].iloc[0])
        ba_dry  = fmt_num(ba_row["BestAppr_Dry"].iloc[0])
    else:
        mlabel=ba_ann=ba_wet=ba_dry="—"

    has_obs = (n_stn > 0)
    row_data = [basin, n_stn, appr,
                obs_ann, obs_wet, obs_dry,
                mlabel,  ba_ann, ba_wet]
    bg_map = {1:CLB, 2:None, 3:abg,
              4:CLB if has_obs else CLR,
              5:CLB if has_obs else CLR,
              6:CLB if has_obs else CLR,
              7:abg, 8:abg, 9:abg}
    for ci,v in enumerate(row_data[:N1],1):
        dat(ws1.cell(row=r,column=ci), v,
            bold=(ci==1), bg=bg_map.get(ci),
            fmt=NUM if isinstance(v,float) else None,
            align=LEFT if ci in (1,3,7) else CENTER)

# Notes
nr = len(basin_order)+5
ws1.merge_cells(f"A{nr}:{get_column_letter(N1)}{nr}")
c = ws1[f"A{nr}"]
c.value = ("Notes: Annual = sum of 12 calendar-month totals averaged over 1995–2014. "
           "Wet = Oct–Mar hydrological season sum averaged over 1995–2014. "
           "Dry = Apr–Sep sum averaged over 1995–2014. "
           "Best Approach determined by composite ranking (Notebook 11, Eq.5). "
           "Blue rows = observed data present | Red = no station data.")
c.font = Font(name="Arial",italic=True,size=8,color="555555")
c.alignment = LEFT

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 2 — All Four Sources
# ══════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("All Four Sources")
N2 = 14
title_row(ws2, f"Basin Mean Precipitation — All Four Sources | {PERIOD_START}–{PERIOD_END}", N2)

grp_header(ws2,2,1,2,"BASIN INFO",CB)
grp_header(ws2,2,3,5,"OBSERVED(Station Mean, mm)",CG)
grp_header(ws2,2,6,8,"BEST SINGLE MODEL (mm)",CB)
grp_header(ws2,2,9,11,"3-MODEL ENSEMBLE (mm)",CO)
grp_header(ws2,2,12,14,"6-MODEL ENSEMBLE (mm)",CP)
ws2.row_dimensions[2].height=22

h2=[("Basin",CB),("NStn",CB),
    ("Annual",CG),("Wet",CG),("Dry",CG),
    ("Annual",CB),("Wet",CB),("Dry",CB),
    ("Annual",CO),("Wet",CO),("Dry",CO),
    ("Annual",CP),("Wet",CP),("Dry",CP)]
for ci,(h,bg) in enumerate(h2,1): hdr(ws2.cell(row=3,column=ci),h,bg=bg)
ws2.row_dimensions[3].height=30
for i,w in enumerate([26,6,11,11,9,11,11,9,11,11,9,11,11,9],1):
    ws2.column_dimensions[get_column_letter(i)].width=w
ws2.freeze_panes="A4"

for ri,basin in enumerate(basin_order):
    r=ri+4
    n_stn = nv(get_basin_val(obs_basin_means, basin,"N_Stations"),0)
    row_data=[
        basin, n_stn,
        fmt_num(get_basin_val(obs_basin_means,  basin,"Annual_Mean")),
        fmt_num(get_basin_val(obs_basin_means,  basin,"Wet_Mean")),
        fmt_num(get_basin_val(obs_basin_means,  basin,"Dry_Mean")),
        fmt_num(get_basin_val(best_basin_means, basin,"Annual_Mean")),
        fmt_num(get_basin_val(best_basin_means, basin,"Wet_Mean")),
        fmt_num(get_basin_val(best_basin_means, basin,"Dry_Mean")),
        fmt_num(get_basin_val(ens3_basin_means, basin,"Annual_Mean")),
        fmt_num(get_basin_val(ens3_basin_means, basin,"Wet_Mean")),
        fmt_num(get_basin_val(ens3_basin_means, basin,"Dry_Mean")),
        fmt_num(get_basin_val(ens6_basin_means, basin,"Annual_Mean")),
        fmt_num(get_basin_val(ens6_basin_means, basin,"Wet_Mean")),
        fmt_num(get_basin_val(ens6_basin_means, basin,"Dry_Mean")),
    ]
    bg_map={1:CLB,2:None,
            3:CLB,4:CLB,5:CLB,
            6:CLG,7:CLG,8:CLG,
            9:CLO,10:CLO,11:CLO,
            12:CLY,13:CLY,14:CLY}
    for ci,v in enumerate(row_data,1):
        dat(ws2.cell(row=r,column=ci),v,
            bold=(ci==1),bg=bg_map.get(ci),
            fmt=NUM if isinstance(v,float) else None,
            align=LEFT if ci==1 else CENTER)

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 3 — Station-Level Detail (all stations, all sources)
# ══════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("Station Detail")
N3 = 13
title_row(ws3,
    f"Station-Level Period Means — All Sources | {PERIOD_START}–{PERIOD_END} (mm)", N3)

grp_header(ws3,2,1,3,"STATION INFO",CB)
grp_header(ws3,2,4,6,"OBSERVED (mm)",CG)
grp_header(ws3,2,7,9,"BEST SINGLE MODEL (mm)",CB)
grp_header(ws3,2,10,12,"3-MODEL ENSEMBLE (mm)",CO)
grp_header(ws3,2,13,13,"BEST MODEL",CB)
ws3.row_dimensions[2].height=16
h3=[("Station ID",CB),("Station Name",CB),("Basin",CB),
    ("Annual",CG),("Wet",CG),("Dry",CG),
    ("Annual",CB),("Wet",CB),("Dry",CB),
    ("Annual",CO),("Wet",CO),("Dry",CO),
    ("BestModel",CB)]
for ci,(h,bg) in enumerate(h3,1): hdr(ws3.cell(row=3,column=ci),h,bg=bg)
ws3.row_dimensions[3].height=30
for i,w in enumerate([10,28,24,11,11,9,11,11,9,11,11,9,18],1):
    ws3.column_dimensions[get_column_letter(i)].width=w
ws3.freeze_panes="A4"

# Join station means from all sources
obs_s  = obs_stn_means[["Station_ID","Station_Name","Basin",
                          "Annual_Mean","Wet_Mean","Dry_Mean"]].copy()
best_s = best_stn_df[["Station_ID","Basin","Annual_Mean",
                        "Wet_Mean","Dry_Mean","Best_Model"]].copy()     if not best_stn_df.empty else pd.DataFrame(
        columns=["Station_ID","Basin","Annual_Mean","Wet_Mean","Dry_Mean","Best_Model"])
ens3_s = ens3_stn_means[["Station_ID","Basin",
                           "Annual_Mean","Wet_Mean","Dry_Mean"]].copy()

obs_s  = obs_s.rename(columns={"Annual_Mean":"Obs_Ann","Wet_Mean":"Obs_Wet","Dry_Mean":"Obs_Dry"})
best_s = best_s.rename(columns={"Annual_Mean":"Sgl_Ann","Wet_Mean":"Sgl_Wet","Dry_Mean":"Sgl_Dry"})
ens3_s = ens3_s.rename(columns={"Annual_Mean":"E3_Ann","Wet_Mean":"E3_Wet","Dry_Mean":"E3_Dry"})

merged_s = obs_s.merge(best_s[["Station_ID","Basin","Sgl_Ann","Sgl_Wet","Sgl_Dry","Best_Model"]],
                        on=["Station_ID","Basin"], how="left")                 .merge(ens3_s[["Station_ID","Basin","E3_Ann","E3_Wet","E3_Dry"]],
                        on=["Station_ID","Basin"], how="left")

# Sort by basin order
basin_order_map = {b:i for i,b in enumerate(basin_order)}
merged_s["sort_key"] = merged_s["Basin"].map(basin_order_map)
merged_s = merged_s.sort_values(["sort_key","Station_ID"]).reset_index(drop=True)

prev_basin = None
for ri,row in merged_s.iterrows():
    r = ri+4
    basin_changed = (row["Basin"] != prev_basin)
    prev_basin    = row["Basin"]
    rbg = CLB if basin_changed else None

    row_data=[row["Station_ID"],row["Station_Name"],row["Basin"],
              fmt_num(row.get("Obs_Ann")), fmt_num(row.get("Obs_Wet")), fmt_num(row.get("Obs_Dry")),
              fmt_num(row.get("Sgl_Ann")), fmt_num(row.get("Sgl_Wet")), fmt_num(row.get("Sgl_Dry")),
              fmt_num(row.get("E3_Ann")),  fmt_num(row.get("E3_Wet")),  fmt_num(row.get("E3_Dry")),
              nv(row.get("Best_Model"))]
    for ci,v in enumerate(row_data,1):
        bg=rbg if ci<=3 else (CLG if 4<=ci<=6 else (CB if 7<=ci<=9 else (CLO if 10<=ci<=12 else None)))
        dat(ws3.cell(row=r,column=ci),v,
            bold=(ci in (1,3)), bg=bg,
            fmt=NUM if isinstance(v,float) else None,
            align=LEFT if ci in (2,3,13) else CENTER)

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 4 — Aridity Context
# ══════════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet("Aridity Context")
N4 = 8
title_row(ws4, f"Basin Means with Aridity Classification | {PERIOD_START}–{PERIOD_END}", N4)

h4=[("Basin",CB),("Aridity Class",CB),("N Stations",CB),
    ("Obs Annual(mm/yr)",CG),("Obs Wet(mm)",CG),("Obs Dry(mm)",CG),
    ("Best ModelAnnual (mm)",CO),("RecommendedApproach",CP)]
for ci,(h,bg) in enumerate(h4,1): hdr(ws4.cell(row=2,column=ci),h,bg=bg)
ws4.row_dimensions[2].height=36
for i,w in enumerate([26,24,10,14,12,11,16,22],1):
    ws4.column_dimensions[get_column_letter(i)].width=w
ws4.freeze_panes="A3"

def aridity(ann):
    if pd.isna(ann) or ann=="—": return "No station data"
    ann=float(ann)
    if ann>=300: return "Humid (>300 mm/yr)"
    if ann>=100: return "Semi-arid (100–300 mm/yr)"
    if ann>=50:  return "Arid (50–100 mm/yr)"
    return "Hyper-arid (<50 mm/yr)"

ARIDITY_BG={"Humid (>300 mm/yr)":"C8E6C9","Semi-arid (100–300 mm/yr)":"FFF9C4",
             "Arid (50–100 mm/yr)":"FFE0B2","Hyper-arid (<50 mm/yr)":"FFCDD2",
             "No station data":CLW}

for ri,basin in enumerate(basin_order):
    r=ri+3
    obs_ann=fmt_num(get_basin_val(obs_basin_means,basin,"Annual_Mean"))
    obs_wet=fmt_num(get_basin_val(obs_basin_means,basin,"Wet_Mean"))
    obs_dry=fmt_num(get_basin_val(obs_basin_means,basin,"Dry_Mean"))
    n_stn  =nv(get_basin_val(obs_basin_means,basin,"N_Stations"),0)
    appr   =recommend_dict.get(basin,"Best Single Model")
    ba_row =best_appr_df[best_appr_df["Basin"]==basin]
    ba_ann =fmt_num(ba_row["BestAppr_Annual"].iloc[0]) if not ba_row.empty else "—"
    arid   =aridity(obs_ann)
    abg    =ARIDITY_BG.get(arid,"F2F2F2")

    row_data=[basin,arid,n_stn,obs_ann,obs_wet,obs_dry,ba_ann,appr]
    for ci,v in enumerate(row_data,1):
        bg=abg if ci<=3 else (CLB if 4<=ci<=6 else (CLG if ci==7 else APPROACH_BG.get(appr,CLY)))
        dat(ws4.cell(row=r,column=ci),v,
            bold=(ci==1),bg=bg,
            fmt=NUM if isinstance(v,float) else None,
            align=LEFT if ci in (1,2,8) else CENTER)

# ── Save workbook ─────────────────────────────────────────────────────────────
out_path = OUTPUT_DIR / "basin_mean_station_vs_best_approach.xlsx"
wb.save(out_path)
print(f"Excel workbook saved: {out_path}")
print("Sheets:")
for s in [ws1.title,ws2.title,ws3.title,ws4.title]:
    print(f"  {s}")


Excel workbook saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\basin mean from station and best approach\basin_mean_station_vs_best_approach.xlsx
Sheets:
  Observed vs Best Approach
  All Four Sources
  Station Detail
  Aridity Context


## 11. Save Master Summary CSV

In [11]:
# Master summary: all four sources in one CSV for downstream use (figures etc.)
master_rows = []
for basin in basin_order:
    appr  = recommend_dict.get(basin,"Best Single Model")
    ba_row = best_appr_df[best_appr_df["Basin"]==basin]
    master_rows.append({
        "Basin"                   : basin,
        "Recommended_Approach"    : appr,
        "Best_Model"              : best_model_dict.get(basin,"?"),
        "N_Obs_Stations"          : nv(get_basin_val(obs_basin_means, basin,"N_Stations"),0),
        "Obs_Annual_mm"           : get_basin_val(obs_basin_means,  basin,"Annual_Mean"),
        "Obs_Wet_mm"              : get_basin_val(obs_basin_means,  basin,"Wet_Mean"),
        "Obs_Dry_mm"              : get_basin_val(obs_basin_means,  basin,"Dry_Mean"),
        "Single_Annual_mm"        : get_basin_val(best_basin_means, basin,"Annual_Mean"),
        "Single_Wet_mm"           : get_basin_val(best_basin_means, basin,"Wet_Mean"),
        "Single_Dry_mm"           : get_basin_val(best_basin_means, basin,"Dry_Mean"),
        "Ens3_Annual_mm"          : get_basin_val(ens3_basin_means, basin,"Annual_Mean"),
        "Ens3_Wet_mm"             : get_basin_val(ens3_basin_means, basin,"Wet_Mean"),
        "Ens3_Dry_mm"             : get_basin_val(ens3_basin_means, basin,"Dry_Mean"),
        "Ens6_Annual_mm"          : get_basin_val(ens6_basin_means, basin,"Annual_Mean"),
        "Ens6_Wet_mm"             : get_basin_val(ens6_basin_means, basin,"Wet_Mean"),
        "Ens6_Dry_mm"             : get_basin_val(ens6_basin_means, basin,"Dry_Mean"),
        "BestAppr_Annual_mm"      : ba_row["BestAppr_Annual"].iloc[0] if not ba_row.empty else np.nan,
        "BestAppr_Wet_mm"         : ba_row["BestAppr_Wet"].iloc[0]    if not ba_row.empty else np.nan,
        "BestAppr_Dry_mm"         : ba_row["BestAppr_Dry"].iloc[0]    if not ba_row.empty else np.nan,
    })

master_df = pd.DataFrame(master_rows)
for col in master_df.select_dtypes(include=float).columns:
    master_df[col] = master_df[col].round(1)

master_csv = OUTPUT_DIR / "master_basin_means_all_sources.csv"
master_df.to_csv(master_csv, index=False)

print(f"Master CSV saved: {master_csv}")
print()
print("All output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        sz = f.stat().st_size / 1024
        print(f"  {f.name:<55} {sz:>7.1f} KB")


Master CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\basin mean from station and best approach\master_basin_means_all_sources.csv

All output files:
  basin_mean_station_vs_best_approach.xlsx                   15.5 KB
  ens3_basin_period_means.csv                                 0.4 KB
  ens3_station_period_means.csv                               3.0 KB
  ens6_basin_period_means.csv                                 0.4 KB
  ens6_station_period_means.csv                               3.0 KB
  master_basin_means_all_sources.csv                          1.8 KB
  obs_basin_period_means.csv                                  0.4 KB
  obs_station_period_means.csv                                3.0 KB
  single_best_basin_period_means.csv                          0.6 KB
  single_best_station_period_means.csv                        3.7 KB
